# 2. ETF Screening and Selection

## Overview
This notebook implements the ETF screening and selection process across four asset classes:

1. **Equities** — distributing UCITS ETFs; filtered by platform, TER, size (>= 100m), and regional beta (>= 0.89).
2. **Bonds** — curated high-liquidity fixed income whitelist.
3. **Precious Metals** — physical gold and silver ETCs benchmarked against 2025 dynamic spot prices.
4. **Commodities** — 3-pillar strategy (**Energy, Agriculture, Industrial Metals**) using high-precision benchmarking (Brent/WTI/Base Metals).

Transitioned from broad indexing to **granular sector exposure** to optimize tracking fidelity and cost-efficiency.


In [2]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import requests
from scipy import stats

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.metrics import calculate_annualized_volatility
from etf_utils.platform_check import check_platform
from etf_utils.database import save_screened_etfs, save_benchmark_etfs

provider = DataProvider()


## Dynamic Benchmark Calculation
We pull live 2025 returns directly from Alpha Vantage to ensure accurate Beta tracking:

| Asset Class | Pillar | Source | Description |
|:---|:---|:---|:---|
| **Equities** | World | `VEVE.L` | developed market baseline |
| **Precious Metals** | Gold/Silver | `GOLD`/`SILVER` | physical spot prices |
| **Commodities** | Energy | `BRENT`/`WTI` | Brent Oil used specifically for **BRNT** |
| **Commodities** | Agriculture | `WHEAT` | soft commodity baseline |
| **Commodities** | Ind Metals | `COPPER`+`ALUM` | Average of base metal spot returns |


In [3]:
provider.provider


'alphavantage'

In [4]:
# Calculate benchmark returns and volatility for 2025
benchmark_year_start = "2025-01-01"
benchmark_year_end = "2025-12-31"

eq_benchmark_symbol  = "VEVE"   # VEVE.L on yfinance
bnd_benchmark_symbol = "SAAA"   # SAAA.L on yfinance
pm_benchmark_symbol  = "SGLN"   # iShares Physical Gold ETC
cmd_benchmark_symbol = "CMOP"   # Invesco Bloomberg Commodity ETC

def _period_metrics(symbol, start, end):
    ret  = round(provider.get_benchmark_period_return(symbol, start, end) * 100, 2)
    yr = provider.get_historical_prices(symbol, start_date=start, end_date=end)["close"]
    vol  = round(calculate_annualized_volatility(yr) * 100, 2)
    sr   = round(ret / vol, 2) if vol > 0 else 0
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _period_metrics(eq_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _period_metrics(bnd_benchmark_symbol, benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _period_metrics(pm_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _period_metrics(cmd_benchmark_symbol, benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")


2025 VEVE  return: 13.81%,  vol: 14.71%,  Sharpe: 0.94
2025 SAAA  return: 2.96%, vol: 4.94%, Sharpe: 0.6
2025 SGLN  return: 53.66%,  vol: 17.65%,  Sharpe: 3.04
2025 CMOP  return: 8.23%, vol: 13.46%, Sharpe: 0.61


In [5]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.L', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TO', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.L', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.L', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.L', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.L', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.L', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.L', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],  
    ['Equity', 'Emerging_APACandEMEA', 'Indonesia', 'EIDO', 'iShares MSCI Indonesia UCITS ETF']
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])

# Calculate 2025 returns for each benchmark ETF using DataProvider
benchmark_df['benchmark_2025_Return'] = benchmark_df['benchmark_ticker'].apply(
    lambda sym: round(provider.get_benchmark_period_return(sym, "2025-01-01", "2025-12-31") * 100, 2)
)
benchmark_df


C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:432: UserWarning: Ticker 'SAUS' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol, start_date=query_start, end_date=end)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:432: UserWarning: Ticker 'XDJP' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol, start_date=query_start, end_date=end)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:432: UserWarning: Ticker 'ISF' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol, start_date=query_start, end_date=end)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:432: UserWarning: Ticker 'CS51' for provider 'alphavant

,asset_class,region,country,benchmark_ticker,benchmark_description,benchmark_2025_Return
10,Equity,Developed_APAC,Australia,SAUS.L,iShares MSCI Australia UCITS ETF,6.23
9,Equity,Developed_APAC,Japan,XDJP.L,Xtrackers Nikkei 225 UCITS ETF 1D,21.04
2,Equity,Developed_AmericasandUK,Canada,ZCN.TO,BMO S&P/TSX Capped Composite,31.51
0,Equity,Developed_AmericasandUK,United Kingdom,ISF.L,iShares Core FTSE 100 UCITS ETF GBP (Dist),25.97
1,Equity,Developed_AmericasandUK,United States,SPY,SPDR S&P 500 ETF Trust,17.72
4,Equity,Developed_EMEA,France,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
3,Equity,Developed_EMEA,Germany,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
5,Equity,Developed_EMEA,Italy,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
7,Equity,Developed_EMEA,Netherlands,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21
6,Equity,Developed_EMEA,Spain,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.21


## ETF Screening Process
We apply multi-layered filters to select the top assets for the **2026-2027** operating year.

### 📈 Equities
- **Filters:** Distributing only, TER <= 0.5%, Size >= 100m, Unhedged.
- **Selection:** Top dividend-yield winner per region + Lowest-TER tracking winner (Beta >= 0.89) from a different country.

### 🏛 Bonds
- **System:** Manually curated high-yield/sovereign whitelist checked for platform availability.

### 🥇 Precious Metals (Granular)
- **Shift:** Moved away from "Mixed Metal" baskets (PHPP) to **pure physical spot trackers** for Gold and Silver.
- **Strict Size:** Enforced 100m AUM minimum (Excludes niche Platinum/Palladium individual trackers).
- **Beta Check:** Funds are compared against live 2025 spot returns to ensure high fidelity to the physical metal.

### 🛢 Commodities (3-Pillar Strategy)
- **Selection:** One winner selected for **Energy, Agriculture, and Industrial Metals** to achieve granular sector exposure.
- **Benchmarking:** 
    - **Energy** benchmarked against live WTI returns.
    - **Agriculture** benchmarked against Wheat spot returns.
    - **Ind Metals** benchmarked against the mean of Copper and Aluminum.
- **Beta Integrity:** Accounts for contango leakage ($0.5 < Beta < 1.5$) while identifying the most cost-efficient tracker for each pillar.


In [6]:
main_df_equities = pd.DataFrame()

for filepath in sorted(DATA_RAW.glob("justetf_class-equity*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)

        if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
            print(f'Skipping {csvl}')
            continue

        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')
        csv_df['platform'] = csv_df['ticker'].apply(check_platform)
        csv_df = csv_df[csv_df['platform'].notna()]

        distributing_df = csv_df.copy()
        distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']
        distributing_df = distributing_df[distributing_df['size'] > 100]
        distributing_df = distributing_df[distributing_df['hedged'] == False]

        #include_tickers = ['IBZL']
        #distributing_df = distributing_df[
        #    (distributing_df['ter'] <= 0.5) | (distributing_df['ticker'].isin(include_tickers))
        #]

        #Tickers to exclude (no london listing)
        remove_tickers = ['NADA']
        distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]



        # Merge benchmark data and compute beta for the entire distributing_df
        distributing_df = pd.merge(
            distributing_df,
            benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2025_Return']],
            on='country', how='left'
        )
        distributing_df["beta"] = (
            distributing_df["2025"] / distributing_df["benchmark_2025_Return"]
        )

        hy_df = (distributing_df.copy()
                 .sort_values(by=['last_year_dividends'], ascending=False)
                 .drop_duplicates(subset=['region_category']))
        hy_df['intra_region_category'] = 'High Yield'

        benchmark_distributing_df = distributing_df.copy()[
            ~distributing_df['country'].isin(hy_df['country'])
        ]

        # Save intermediate benchmark data
        save_benchmark_etfs(benchmark_distributing_df, asset_class='equity', portfolio_year=2026)

        # Tolerance adjusted to 0.99 to allow valid ETFs to pass minor data feed discrepancies
        benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 0.89]
        beta_df = (benchmark_distributing_df.copy()
                   .sort_values(by=['ter', 'beta'], ascending=[True, False])
                   .drop_duplicates(subset=['region_category']))
        beta_df['intra_region_category'] = 'Beta'

        main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')



# Fetch latest close price via DataProvider (yfinance, no API key)
def _get_price(ticker):
    try:
        return provider.get_latest_price(ticker)
    except Exception as e:
        print(f"Price error for {ticker}: {e}")
        return None, None

if not main_df_equities.empty:
    main_df_equities[['yday_close_price_date', 'yday_close_price']] = (
        main_df_equities['ticker'].apply(_get_price).to_list()
    )
else:
    main_df_equities['yday_close_price_date'] = []
    main_df_equities['yday_close_price'] = []

save_screened_etfs(main_df_equities, 'equity', portfolio_year=2026)
print(f"Saved {len(main_df_equities)} equity ETFs")
main_df_equities[main_df_equities["platform"].notna()].groupby(["region_category","intra_region_category"]).sum()

Processing equity file for developed_americasanduk: justetf_class-equity_developed_americasanduk.csv
Processing equity file for developed_apac: justetf_class-equity_developed_apac.csv
Processing equity file for developed_emea: justetf_class-equity_developed_emea.csv
Processing equity file for emerging_americas: justetf_class-equity_emerging_americas.csv
Processing equity file for emerging_apacandemea: justetf_class-equity_emerging_apacandemea.csv


C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'JEQP' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'ISJP' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'VGER' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'HIDR' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_pri

Saved 8 equity ETFs


C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'HKOR' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)


wkn ticker        valor  \
region_category         intra_region_category                               
Developed_APAC          High Yield             A0Q1YX   ISJP    4218299.0   
Developed_AmericasandUK Beta                   LYX0YA   LCUK   40587057.0   
                        High Yield             A40FFF   JEQP  138031097.0   
Developed_EMEA          Beta                   A2JF6S   VGER   41860972.0   
                        High Yield             A0MZWP   IMIB    3246482.0   
Emerging_APACandEMEA    Beta                   A1JJU5   HKOR   12843012.0   
                        High Yield             A1H8BN   HIDR   12789497.0   
Emerging_Americas       High Yield             A0HGWA   IBZL    2308866.0   

                                                                                            name  \
region_category         intra_region_category                                                      
Developed_APAC          High Yield                 iShares MSCI Japan Small Cap UCITS ETF (Dist)   
Developed_AmericasandUK Beta                             Amundi UK Equity All Cap UCITS ETF Dist   
                        High Yield             JPMorgan Nasdaq Equity Premium Income Active U...   
Developed_EMEA          Beta                   Vanguard Germany All Cap UCITS ETF (EUR) Distr...   
                        High Yield                         iShares FTSE MIB UCITS ETF EUR (Dist)   
Emerging_APACandEMEA    Beta                                HSBC MSCI Korea Capped UCITS ETF USD   
                        High Yield                             HSBC MSCI Indonesia UCITS ETF USD   
Emerging_Americas       High Yield                          iShares MSCI Brazil UCITS ETF (Dist)   

                                                    inception_date  \
region_category         intra_region_category                        
Developed_APAC          High Yield             2008-05-09 00:00:00   
Developed_AmericasandUK Beta                   2018-02-27 00:00:00   
                        High Yield             2024-10-29 00:00:00   
Developed_EMEA          Beta                   2018-07-17 00:00:00   
                        High Yield             2007-07-06 00:00:00   
Emerging_APACandEMEA    Beta                   2011-04-06 00:00:00   
                        High Yield                      2011-03-28   
Emerging_Americas       High Yield             2005-11-18 00:00:00   

                                               age_in_days  age_in_years  \
region_category         intra_region_category                              
Developed_APAC          High Yield                    6546     17.934247   
Developed_AmericasandUK Beta                          2965      8.123288   
                        High Yield                     529      1.449315   
Developed_EMEA          Beta                          2825      7.739726   
                        High Yield                    6854     18.778082   
Emerging_APACandEMEA    Beta                          5484     15.024658   
                        High Yield                    5493     15.049315   
Emerging_Americas       High Yield                    7449     20.408219   

                                                        strategy  \
region_category         intra_region_category                      
Developed_APAC          High Yield                     Long-only   
Developed_AmericasandUK Beta                           Long-only   
                        High Yield             Long-only, Active   
Developed_EMEA          Beta                           Long-only   
                        High Yield                     Long-only   
Emerging_APACandEMEA    Beta                           Long-only   
                        High Yield                     Long-only   
Emerging_Americas       High Yield                     Long-only   

                                              domicile_country currency  ...  \
region_category         intra_region_category          

In [7]:
############# BONDS #############
main_df_bonds = pd.DataFrame()

# Manually curated bond ETF list
bonds_data = {
    'asset_class': ['bonds'] * 8,
    'region_category': [
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA',
        'Emerging_APACandEMEA',
    ],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP'],
}
df_bonds_list = pd.DataFrame(bonds_data)

for filepath in sorted(DATA_RAW.glob("justetf_class-bonds_*.csv")):
    csvl = filepath.name
    try:
        csv_df = pd.read_csv(filepath)
        if csv_df.empty: continue
        
        # Filter raw data for our whitelist first
        for _, row in df_bonds_list.iterrows():
            if row['ticker'] in csv_df['ticker'].values:
                ticker = row['ticker']
                csv_df_ticker = csv_df[csv_df['ticker'] == ticker].copy()
                
                # [REVERTED] Platform check AFTER whitelist check
                csv_df_ticker['platform'] = check_platform(ticker, name=csv_df_ticker['name'].values[0])
                if pd.isna(csv_df_ticker['platform'].values[0]): continue
                
                csv_df_ticker['intra_region_category'] = row['intra_region_category']
                csv_df_ticker['region_category'] = row['region_category']
                main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

if not main_df_bonds.empty:
    main_df_bonds[['yday_close_price_date', 'yday_close_price']] = main_df_bonds['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_bonds, 'bonds', portfolio_year=2026)
    print(f"Saved {len(main_df_bonds)} bond ETFs")
    display(main_df_bonds[['ticker', 'name', 'intra_region_category', 'ter', 'platform']])


Saved 7 bond ETFs


,ticker,name,intra_region_category,ter,platform
0,IGLT,iShares Core UK Gilts UCITS ETF,Govt,0.07,InvestEngine
1,SLXX,iShares Core GBP Corporate Bond UCITS ETF,Corp,0.20,InvestEngine
2,TRXG,Invesco US Treasury Bond 7-10 Year UCITS ETF Dist,Govt,0.06,InvestEngine
3,UC81,UBS BBG US Liquid Corp 1-5 UCITS ETF USD dis,Corp,0.16,InvestEngine
4,EMCP,iShares J.P. Morgan USD EM Corporate Bond UCIT...,Corp,0.50,InvestEngine
5,PRIR,Amundi Prime Euro Government Bond UCITS ETF Dist,Govt,0.05,InvestEngine
6,VECP,Vanguard EUR Corporate Bond UCITS ETF (EUR) Di...,Corp,0.07,InvestEngine


In [8]:
############# PRECIOUS METALS & COMMODITIES #############

def _metal_type(name: str) -> str:
    n = str(name).lower()
    if 'silver' in n: return 'silver'
    if 'gold' in n: return 'gold'
    return 'other'

def _get_pm_beta(row):
    m = row['metal_type']
    if m in metal_benchmarks and not pd.isna(metal_benchmarks[m]):
        return (row.get('2025', 0) / 100.0) / metal_benchmarks[m]
    return 1.0

def _commodity_category(row) -> str:
    name = str(row['name']).lower()
    ticker = str(row['ticker'])
    if any(x in name for x in ['ex-', 'ex ', 'without']): return 'Broad'
    if ticker == 'NRGT': return 'Broad' 
    if any(x in name for x in ['energy', 'oil', 'petrol', 'fuel']): return 'Energy'
    if any(x in name for x in ['agri', 'wheat', 'corn', 'grain', 'softs']): return 'Agriculture'
    if any(x in name for x in ['metal', 'copper', 'alu', 'zinc', 'nickel']): return 'Industrial Metals'
    return 'Broad'

def _compute_cmd_beta(row):
    ticker = str(row['ticker'])
    cat = row['intra_region_category']
    if cat == 'Energy':
        bench = brent_ret if 'BRNT' in ticker else wti_ret
    elif cat == 'Agriculture':
        bench = agri_ret
    elif cat == 'Industrial Metals':
        bench = (copper_ret + alum_ret) / 2.0
    else:
        return 1.0
    if bench and bench != 0:
        return (row.get('2025', 0) / 100.0) / bench
    return 1.0

# --- Precious Metals ---
pm_raw_path = DATA_RAW / 'justetf_class-preciousMetals_global.csv'
main_df_pm = pd.DataFrame()

if pm_raw_path.exists():
    pm_df = pd.read_csv(pm_raw_path)
    pm_df['asset_class'] = 'preciousMetals'
    pm_df['region_category'] = 'Global'
    pm_df['metal_type'] = pm_df['name'].apply(_metal_type)
    
    pm_filtered = pm_df[
        (pm_df['size'] > 100) & (pm_df['ter'] < 0.6) & (pm_df['hedged'] == False)
    ].copy()

    # Platform check after filtering
    pm_filtered['platform'] = pm_filtered.apply(lambda row: check_platform(row['ticker'], name=row['name']), axis=1)
    pm_filtered = pm_filtered[pm_filtered['platform'].notna()]

    #Tickers to exclude (no london listing)
    remove_tickers = ['4GLD']
    pm_filtered = pm_filtered[~pm_filtered['ticker'].isin(remove_tickers)]

    print("Fetching dynamic precious metal benchmarks...")
    metal_benchmarks = {
        'gold': provider.get_benchmark_period_return('GOLD', start='2024-12-31', end='2025-12-31'),
        'silver': provider.get_benchmark_period_return('SILVER', start='2024-12-31', end='2025-12-31'),
    }
    pm_filtered['beta'] = pm_filtered.apply(_get_pm_beta, axis=1)
    
    main_df_pm = pm_filtered[pm_filtered['metal_type'].isin(['gold', 'silver'])].sort_values(['ter', 'size']).groupby('metal_type').first().reset_index()
    main_df_pm['intra_region_category'] = main_df_pm['metal_type'].str.capitalize()
    main_df_pm[['yday_close_price_date', 'yday_close_price']] = main_df_pm['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_pm, 'preciousMetals', portfolio_year=2026)

# --- Commodities ---
print("Fetching dynamic sector benchmarks...")
wti_ret    = provider.get_benchmark_period_return('WTI', start='2024-12-31', end='2025-12-31')
brent_ret  = provider.get_benchmark_period_return('BRENT', start='2024-12-31', end='2025-12-31')
agri_ret   = provider.get_benchmark_period_return('WHEAT', start='2024-12-31', end='2025-12-31')
copper_ret = provider.get_benchmark_period_return('COPPER', start='2024-12-31', end='2025-12-31')
alum_ret   = provider.get_benchmark_period_return('ALUMINUM', start='2024-12-31', end='2025-12-31')

cmd_raw_path = DATA_RAW / 'justetf_class-commodities_global.csv'
main_df_cmd = pd.DataFrame()

if cmd_raw_path.exists():
    cmd_df = pd.read_csv(cmd_raw_path)
    cmd_df['asset_class'] = 'commodities'
    cmd_df['region_category'] = 'Global'
    cmd_df['intra_region_category'] = cmd_df.apply(_commodity_category, axis=1)
    
    cmd_filtered = cmd_df[(cmd_df['size'] > 100) & (cmd_df['ter'] < 0.6) & (cmd_df['hedged'] == False)].copy()
    cmd_filtered['platform'] = cmd_filtered.apply(lambda row: check_platform(row['ticker'], name=row['name']), axis=1)
    cmd_filtered = cmd_filtered[cmd_filtered['platform'].notna()]

    cmd_filtered['beta'] = cmd_filtered.apply(_compute_cmd_beta, axis=1)
    main_df_cmd = cmd_filtered[cmd_filtered['intra_region_category'].isin(['Energy', 'Agriculture', 'Industrial Metals'])].sort_values(['ter', 'size']).groupby('intra_region_category').first().reset_index()
    main_df_cmd[['yday_close_price_date', 'yday_close_price']] = main_df_cmd['ticker'].apply(_get_price).to_list()
    save_screened_etfs(main_df_cmd, 'commodities', portfolio_year=2026)

# --- Summary ---
dfs = [df for df in [main_df_equities, main_df_bonds, main_df_pm, main_df_cmd] if not df.empty]
if dfs:
    summary_df = pd.concat(dfs, ignore_index=True)
    display(summary_df[['asset_class', 'region_category', 'intra_region_category', 'ticker', 'name', 'ter', 'beta', 'platform']])


Fetching dynamic precious metal benchmarks...
Fetching commodity GOLD from AlphaVantage via GOLD_SILVER_HISTORY...
Fetching commodity SILVER from AlphaVantage via GOLD_SILVER_HISTORY...
Price error for XGDP: No AlphaVantage data for 'XGDP.LON'. Check API key and symbol.


C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'WSIL' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)


Fetching dynamic sector benchmarks...
Fetching commodity WTI from AlphaVantage via WTI...
Fetching commodity BRENT from AlphaVantage via BRENT...
Fetching commodity WHEAT from AlphaVantage via WHEAT...
Fetching commodity COPPER from AlphaVantage via COPPER...
Fetching commodity ALUMINUM from AlphaVantage via ALUMINUM...
Price error for WEAP: No AlphaVantage data for 'WEAP.LON'. Check API key and symbol.


C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'AIGE' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)
C:\Users\rakes\My Drive\dev\etf-isa-portfolio\etf_utils\data_provider.py:409: UserWarning: Ticker 'NICK' for provider 'alphavantage' not found in currency_units.json. Falling back to heuristic pence detection. 
  df = self.get_historical_prices(symbol)


,asset_class,region_category,intra_region_category,ticker,name,ter,beta,platform
0,equity,Developed_AmericasandUK,High Yield,JEQP,JPMorgan Nasdaq Equity Premium Income Active U...,0.35,0.416479,InvestEngine
1,equity,Developed_AmericasandUK,Beta,LCUK,Amundi UK Equity All Cap UCITS ETF Dist,0.04,0.951097,InvestEngine
2,equity,Developed_APAC,High Yield,ISJP,iShares MSCI Japan Small Cap UCITS ETF (Dist),0.58,0.961502,InvestEngine
3,equity,Developed_EMEA,High Yield,IMIB,iShares FTSE MIB UCITS ETF EUR (Dist),0.35,1.581000,InvestEngine
4,equity,Developed_EMEA,Beta,VGER,Vanguard Germany All Cap UCITS ETF (EUR) Distr...,0.07,0.968451,InvestEngine
5,equity,Emerging_Americas,High Yield,IBZL,iShares MSCI Brazil UCITS ETF (Dist),0.74,0.930251,InvestEngine
6,equity,Emerging_APACandEMEA,High Yield,HIDR,HSBC MSCI Indonesia UCITS ETF USD,0.50,-2.026423,InvestEngine
7,equity,Emerging_APACandEMEA,Beta,HKOR,HSBC MSCI Korea Capped UCITS ETF USD,0.50,0.894853,InvestEngine
8,NaN,Developed_AmericasandUK,Govt,IGLT,iShares Core UK Gilts UCITS ETF,0.07,NaN,InvestEngine
9,NaN,Developed_AmericasandUK,Corp,SLXX,iShares Core GBP Corporate Bond UCITS ETF,0.20,NaN,InvestEngine
